In [3]:
import pandas as pd
df = pd.read_csv("EMP_TEMP_SEX_GAI_NB_A-20260911T1441.csv.gz")
df.head(10)

,ref_area,source,indicator,sex,classif1,time,obs_value,obs_status,note_indicator,note_source
0,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_TOTAL,2021,7679.474,NaN,NaN,R1:3513_S3:8
1,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_4,2021,2.385,U,NaN,R1:3513_S3:8
2,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_3,2021,29.094,NaN,NaN,R1:3513_S3:8
3,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_2,2021,720.246,NaN,NaN,R1:3513_S3:8
4,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_1,2021,329.742,NaN,NaN,R1:3513_S3:8
5,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_5,2021,258.223,NaN,NaN,R1:3513_S3:8
6,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_T,GAI_GRADIENT_X,2021,6339.784,NaN,NaN,R1:3513_S3:8
7,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_M,GAI_GRADIENT_TOTAL,2021,5850.428,NaN,NaN,R1:3513_S3:8
8,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_M,GAI_GRADIENT_4,2021,2.385,U,NaN,R1:3513_S3:8
9,AFG,BA:15715,EMP_TEMP_SEX_GAI_NB,SEX_M,GAI_GRADIENT_3,2021,29.094,NaN,NaN,R1:3513_S3:8


In [ ]:
import requests
import pandas as pd

IMF_DATAMAPPER_BASE = "https://www.imf.org/external/datamapper/api/v1/"

AIPI_INDICATORS = {
    "AI_PI": "aipi_overall",
    "DI": "aipi_digital_infrastructure",
    "IEI": "aipi_innovation_econ_integration",
    "HCLMP": "aipi_human_capital_labor_policy",
    "RE": "aipi_regulation_ethics",
}

SESSION = requests.Session()


def download_aipi_indicator(code, session=SESSION):
    url = f"{IMF_DATAMAPPER_BASE}{code}"

    response = session.get(url, timeout=60)
    response.raise_for_status()

    payload = response.json()
    values = payload.get("values", {})

    if not values:
        print(f"Warning: no data returned for IMF code '{code}'. The endpoint may not publish that indicator.")
        return pd.DataFrame(columns=["iso3", "year", code])

    indicator_values = values.get(code)
    if indicator_values is None:
        for candidate in values.values():
            if isinstance(candidate, dict):
                maybe = candidate.get(code)
                if maybe is not None:
                    indicator_values = maybe
                    break

    if indicator_values is None:
        print(f"Warning: IMF payload for '{code}' is not in the expected structure.")
        return pd.DataFrame(columns=["iso3", "year", code])

    rows = []
    for iso3, year_map in indicator_values.items():
        if not isinstance(year_map, dict):
            continue
        for year, value in year_map.items():
            rows.append({
                "iso3": iso3,
                "year": int(year),
                code: value
            })

    return pd.DataFrame(rows)


# Download all AIPI indicators
aipi_frames = {
    key: download_aipi_indicator(value)
    for key, value in AIPI_INDICATORS.items()
}

print("AIPI data downloaded successfully.")

for code, df in aipi_frames.items():
    print(f"{code}: {len(df):,} rows, {df['iso3'].nunique() if 'iso3' in df.columns else 0} countries")

KeyError: 'values'

In [ ]:
import requests
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# WORLD BANK WDI — GDP PER CAPITA (CURRENT US$)
# Indicator: NY.GDP.PCAP.CD
# ---------------------------------------------------------

WB_API_BASE = "https://api.worldbank.org/v2"
GDP_INDICATOR = "NY.GDP.PCAP.CD"

# Use the active notebook working directory instead of __file__
ROOT = Path.cwd()
RAW_WB = ROOT / "data" / "raw" / "world_bank"
RAW_WB.mkdir(parents=True, exist_ok=True)


def download_world_bank_indicator(indicator, output_file):
    """
    Download a World Bank WDI indicator for all countries/years
    and save the raw API response as a CSV.
    """

    url = f"{WB_API_BASE}/country/all/indicator/{indicator}"

    params = {
        "format": "json",
        "per_page": 20000
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()

    if not isinstance(data, list) or len(data) < 2:
        raise ValueError(f"Unexpected World Bank API response for {indicator}: {data}")

    observations = data[1]
    rows = []

    for obs in observations:
        rows.append({
            "iso3": obs.get("countryiso3code"),
            "country": obs.get("country", {}).get("value"),
            "year": int(obs["date"]),
            "gdp_pc_current_usd": obs.get("value")
        })

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No observations returned for {indicator}.")

    df = df[df["iso3"].notna()].copy()
    df["gdp_pc_current_usd"] = pd.to_numeric(
        df["gdp_pc_current_usd"],
        errors="coerce"
    )

    df.to_csv(output_file, index=False)

    return df


gdp_file = RAW_WB / "world_bank_gdp_per_capita.csv"

gdp_df = download_world_bank_indicator(
    GDP_INDICATOR,
    gdp_file
)

print("\nWorld Bank GDP per capita downloaded.")
print(f"Rows: {len(gdp_df):,}")
print(f"Countries: {gdp_df['iso3'].nunique():,}")
print(f"Years: {gdp_df['year'].min()}–{gdp_df['year'].max()}")
print(f"Saved to: {gdp_file}")

NameError: name '__file__' is not defined